# Blue team: build private origin-labelled data
Run with **Python (botfluencer CUDA)**, from a private copy outside OneDrive/the repo.
The negative class is a confirmed NPC origin, **not human-written text**. NPCs may use language models too.
The positive class here is synthetic competitor-style Qwen text, not verified competitor ground truth.
We include Yes and No equally, group all examples from a source author together, and remove duplicate text before splitting.
Never label an unknown live author as a player merely because it arrived after opening.
The next notebook must pass independent author-held-out evaluation; synthetic results alone cannot enable automatic reporting.


In [ ]:
from pathlib import Path

from botfluencer.blue.data import private_directory

# Copy this notebook into ~/botfluencer-notebooks before running it.
private_directory(Path.cwd())
DATA = Path("/mnt/d/botfluencer/data/blue-v1")
RELEASES = Path("/mnt/d/botfluencer/models/releases")
ACTIVE = Path("/mnt/d/botfluencer/models/blue-active")

In [ ]:
from botfluencer.blue.data import export_npcs, generate_players, prepare_rows, read_jsonl
from botfluencer.config import Settings

settings = Settings(_env_file=Path.home() / ".config/botfluencer/runtime.env")
# Read-only export of the existing team-confirmed NPC archive; no CTN API calls.
await export_npcs(settings, DATA)

## Generate optional training examples
Each example consumes one competition-model request. 150 per side is 300 requests and takes at least about 100 minutes at 3/minute, longer if teammates share the quota.
This uses the organiser-provided Qwen endpoint, never the social publishing API. The output is resumable; rerun with the same count after interruption.
Coordinate with teammates before using this quota. Increase sample count/diversity before expecting useful results.


In [ ]:
RUN_SYNTHESIS = False  # Change to True when ready to spend the shared model quota.
PER_SIDE = 150
if RUN_SYNTHESIS:
    network_settings = settings.model_copy(update={"ctn_network_enabled": True})
    synthesis_summary = await generate_players(network_settings, DATA, per_side=PER_SIDE)
    print(synthesis_summary)
else:
    print("Synthesis disabled; set RUN_SYNTHESIS=True to generate examples.")

In [ ]:
from collections import Counter

rows = prepare_rows(read_jsonl(DATA / "npc.jsonl") + read_jsonl(DATA / "synthetic.jsonl"))
print("Counts (split, label):", dict(Counter((r["split"], r["label"]) for r in rows)))
print("Synthetic sides:", dict(Counter(r.get("side") for r in rows if r["label"] == 1)))
groups = {s: {r["group"] for r in rows if r["split"] == s} for s in ["train", "validation", "test"]}
assert not groups["train"] & groups["validation"]
assert not groups["train"] & groups["test"]
assert not groups["validation"] & groups["test"]
print("Author/context groups are disjoint. Raw content remains private.")